In [2]:
from google.colab import userdata
SARVAM_API_KEY = userdata.get('SARVAM_API_KEY')

In [1]:
!pip install -q sarvamai pydub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.3/269.3 kB 6.8 MB/s eta 0:00:00


In [5]:
from sarvamai import SarvamAI
import base64
from google.colab import userdata

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

# Code-mixed Kannada sentence a real user might say
sample_text = "Nanu office ge hogbekittu, but bus late aagide. Meeting yavaga start aaguttade?"

tts_response = client.text_to_speech.convert(
    text=sample_text,          # ← was 'inputs=[...]', now just 'text='
    target_language_code="kn-IN",
    speaker="anushka"
)

# Save the audio
audio_data = base64.b64decode(tts_response.audios[0])
audio_path = "sample_codemix_kannada.wav"
with open(audio_path, "wb") as f:
    f.write(audio_data)

print(f'Sample audio created: {audio_path}')
print(f'Text spoken: "{sample_text}"')

from IPython.display import Audio, display
display(Audio(audio_path))

Sample audio created: sample_codemix_kannada.wav
Text spoken: "Nanu office ge hogbekittu, but bus late aagide. Meeting yavaga start aaguttade?"


In [11]:
from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

def transcribe(audio_path, mode):
    with open(audio_path, 'rb') as f:
        response = client.speech_to_text.transcribe(
            file=f,
            model='saaras:v3',
            mode=mode,
            language_code='kn-IN'  # change to hi-IN, ta-IN etc. for other languages
        )
    return response.transcript

print('🎙️ Transcribing...')
transcript      = transcribe(audio_path, 'transcribe')
translation     = transcribe(audio_path, 'translate')
codemix         = transcribe(audio_path, 'codemix')

print(f'\n Transcribe (original language):\n   {transcript}')
print(f'\n Translate (English):\n   {translation}')
print(f'\n Codemix (mixed script preserved):\n   {codemix}')

🎙️ Transcribing...

 Transcribe (original language):
   ಯಾನಿ ಆಫೀಸ್ ವಾ ಹಾವ್ ಬಿಕೆಟ್ಟು ಬಟ್ ಬಸ್ ಲೇಟ್ ಅಜೆಡ್. ಲೀಟಿಂಗ್ ಯಾವಾಗ ಸ್ಟಾರ್ಟ್ ಅಗಟೆಡ್?

 Translate (English):
   The office is closed, but the bus is late. When will the meeting start?

 Codemix (mixed script preserved):
   ಯಾನಿ office ವಾ ಹವ್ ಬಿಕೆಟ್ಟು ಬಟ್ ಬಸ್ ಲೇಟ್ ಅಗೆಡ್. ಲೀಟಿಂಗ್ ಯಾವಾಗ start ಅಗೆಡ್ ಟೆಡ್.


In [12]:
# Demonstrate the retrieval gap
print('=' * 60)
print('THE CODE-MIXED RETRIEVAL PROBLEM')
print('=' * 60)
print(f'\nUser query (codemix transcript):')
print(f'  "{codemix}"')
print(f'\nThis query contains:')

kannada_words = [w for w in codemix.split() if any('\u0C80' <= c <= '\u0CFF' for c in w)]
english_words = [w for w in codemix.split() if all(c.isascii() or not c.isalpha() for c in w) and w.isalpha()]

print(f'  Kannada script tokens : {kannada_words}')
print(f'  Latin script tokens   : {english_words}')
print(f'\nIf your knowledge base is in English:')
print(f'  Query using codemix   → partial match only')
print(f'  Query using translate → "{translation}"')
print(f'   Use the TRANSLATED query for English knowledge bases')
print(f'\nThis is why translate mode is the right retrieval key.')

THE CODE-MIXED RETRIEVAL PROBLEM

User query (codemix transcript):
  "ಯಾನಿ office ವಾ ಹವ್ ಬಿಕೆಟ್ಟು ಬಟ್ ಬಸ್ ಲೇಟ್ ಅಗೆಡ್. ಲೀಟಿಂಗ್ ಯಾವಾಗ start ಅಗೆಡ್ ಟೆಡ್."

This query contains:
  Kannada script tokens : ['ಯಾನಿ', 'ವಾ', 'ಹವ್', 'ಬಿಕೆಟ್ಟು', 'ಬಟ್', 'ಬಸ್', 'ಲೇಟ್', 'ಅಗೆಡ್.', 'ಲೀಟಿಂಗ್', 'ಯಾವಾಗ', 'ಅಗೆಡ್', 'ಟೆಡ್.']
  Latin script tokens   : ['office', 'start']

If your knowledge base is in English:
  Query using codemix   → partial match only
  Query using translate → "The office is closed, but the bus is late. When will the meeting start?"
   Use the TRANSLATED query for English knowledge bases

This is why translate mode is the right retrieval key.


In [17]:
context = """
Company Meeting Schedule:
- Daily standup: 10:00 AM IST
- Weekly team meeting: Monday 11:00 AM IST
- Bus shuttle timings: 9:00 AM, 9:30 AM, 10:00 AM from main gate
- If you're running late, notify your manager on Slack #delays channel
"""

response = client.chat.completions(
    model="sarvam-105b",
    messages=[
        {"role": "system", "content": f"You are a helpful office assistant. Answer based on this context:\n{context}"},
        {"role": "user", "content": translation}
    ]
)

print("Answer:")
print(response.choices[0].message.content)

Answer:
Based on the schedule provided, the daily standup is at **10:00 AM IST**.

The information doesn't state that the meeting will be delayed because the bus is late. However, it does instruct you to notify your manager on the Slack #delays channel if you are running late.


In [16]:
# Supported language codes
SUPPORTED_LANGUAGES = {
    'kn-IN': 'Kannada',
    'hi-IN': 'Hindi',
    'ta-IN': 'Tamil',
    'te-IN': 'Telugu',
    'ml-IN': 'Malayalam',
    'mr-IN': 'Marathi',
    'bn-IN': 'Bengali',
    'gu-IN': 'Gujarati',
    'pa-IN': 'Punjabi',
    'or-IN': 'Odia',
}

print('Supported languages for Saaras v3:')
for code, name in SUPPORTED_LANGUAGES.items():
    print(f'  {code}  →  {name}')

print('\nTo use a different language, change language_code in Step 4.')
print('Or set language_code="auto" to let Saaras v3 detect automatically.')

Supported languages for Saaras v3:
  kn-IN  →  Kannada
  hi-IN  →  Hindi
  ta-IN  →  Tamil
  te-IN  →  Telugu
  ml-IN  →  Malayalam
  mr-IN  →  Marathi
  bn-IN  →  Bengali
  gu-IN  →  Gujarati
  pa-IN  →  Punjabi
  or-IN  →  Odia

To use a different language, change language_code in Step 4.
Or set language_code="auto" to let Saaras v3 detect automatically.
